# Backdoor Toolbox — Kaggle Runner

**執行順序：** Cell 1 → Cell 2 → **Cell 3（選擇 attack）** → Cell 4

> **前置條件：** Kaggle Notebook 設定中請開啟 `Internet on` 以便 clone repo 與下載資料集。

## Cell 1 — Clone Repo

In [ ]:
import os

REPO_URL = 'https://github.com/your-username/backdoor-toolbox.git'  # ← 改成你的 repo URL
ROOT     = '/kaggle/working/backdoor-toolbox'

if not os.path.isdir(ROOT):
    os.system(f'git clone {REPO_URL} {ROOT}')
    print('Clone 完成')
else:
    print('Repo 已存在，跳過 clone')

os.chdir(ROOT)
print(f'Working directory: {os.getcwd()}')

## Cell 2 — Import & 顯示目前設定

In [ ]:
import time
import tomllib
from generator import phase1, phase2, phase3, phase_other_attack

with open('config.toml', 'rb') as f:
    cfg = tomllib.load(f)['Trainer']

print('=== 目前 config.toml 設定 ===')
for k, v in cfg.items():
    if not isinstance(v, dict):
        print(f'  {k:15s} = {v}')

## Cell 4 — 選擇要訓練的 Attack

取消註解想跑的 attack，其餘保持註解。

| 類型 | 可選項目 |
|------|----------|
| Poisoning attacks | `none` `badnet` `blend` `trojan` `SIG` `dynamic` `ISSBA` `WaNet` `TaCT` `refool` `adaptive_blend` `adaptive_patch` `adaptive_k_way` `badnet_all_to_all` `SleeperAgent` `clean_label` |
| Other attacks | `bpp` `trojannn` `BadEncoder` `SRA` `WB` |

In [ ]:
# cfg['dataset']      = 'cifar10'
# cfg['num_models']   = 15
# cfg['poison_rate']  = 0.1
# cfg['data_rate']    = 1
# cfg['train_source'] = 'train'
# cfg['clean_budget'] = 2000

# cfg['WaNet']['cover_rate']          = 0.1
# cfg['TaCT']['cover_rate']           = 0.005
# cfg['adaptive_blend']['cover_rate'] = 0.003
# cfg['adaptive_patch']['cover_rate'] = 0.006

print('=== 實際使用的設定 ===')
for k, v in cfg.items():
    if not isinstance(v, dict):
        print(f'  {k:15s} = {v}')

## Cell 5 — 執行

In [ ]:
POISON_TYPES = [
    # ── Poisoning attacks ──────────────────────────────────────────────
    # 'none',
    # 'badnet',
    # 'blend',
    # 'trojan',
    # 'SIG',
    # 'dynamic',
    # 'ISSBA',
    # 'WaNet',
    # 'TaCT',
    # 'refool',
    # 'adaptive_blend',
    # 'adaptive_patch',
    # 'adaptive_k_way',
    # 'badnet_all_to_all',
    # 'SleeperAgent',
    # 'clean_label',

    # ── Other attacks ──────────────────────────────────────────────────
    # 'bpp',
    # 'trojannn',
    # 'BadEncoder',
    # 'SRA',
    # 'WB',
]

OTHER_ATTACKS = {'bpp', 'trojannn', 'BadEncoder', 'SRA', 'WB'}

assert len(POISON_TYPES) > 0, '至少選一個 attack！'
print(f'選擇的 attacks：{POISON_TYPES}')

## Cell 4 — 執行

In [ ]:
def run_process(cfg, poison_type):
    dataset      = cfg['dataset']
    data_rate    = cfg['data_rate']
    poison_rate  = cfg['poison_rate']
    num_models   = cfg['num_models']
    train_source = cfg['train_source']
    clean_budget = cfg['clean_budget']
    cover_rate   = None

    phase1(dataset, clean_budget)

    if poison_type in OTHER_ATTACKS:
        phase_other_attack(
            dataset, data_rate, poison_type, poison_rate,
            num_models, train_source, clean_budget, cover_rate,
        )
        return

    if poison_type in ('WaNet', 'TaCT', 'adaptive_blend', 'adaptive_patch'):
        cover_rate = cfg[poison_type]['cover_rate']

    phase2(
        dataset, data_rate, poison_type, poison_rate,
        train_source, clean_budget, cover_rate,
    )
    phase3(
        dataset, data_rate, poison_type, poison_rate,
        num_models, train_source, clean_budget, cover_rate,
    )


total     = len(POISON_TYPES)
t_overall = time.time()

for idx, poison_type in enumerate(POISON_TYPES, 1):
    print(f'\n{"="*50}')
    print(f'[{idx}/{total}] poison_type = {poison_type}')
    print(f'{"="*50}')
    t0 = time.time()
    run_process(cfg, poison_type)
    elapsed = time.time() - t0
    print(f'[{idx}/{total}] {poison_type} 完成，耗時 {elapsed/60:.1f} 分鐘')

total_elapsed = time.time() - t_overall
print(f'\n全部完成，總耗時 {total_elapsed/60:.1f} 分鐘')